# Business Entity Resolution: Kaggle runner

**Setup (once):** add the competition data as an input dataset (folders `train/` and `test/`), Settings → **Internet on**, Accelerator **None (CPU)**, then set `DATA_SLUG`.

**How to run (important):** use **Save Version → Save & Run All (Commit)**. It runs headless for up to 12 h and does **not** need the browser open. Interactive *draft* sessions get stopped when the browser is idle, which wipes `/kaggle/working`. The full pipeline takes ~3 h on 4 CPU cores. When the commit finishes, download `output/matching_results.tsv` from the version's **Output** tab.

Stages skip themselves if their outputs exist, and test inference checkpoints every batch, so a restarted draft session resumes instead of starting over.

In [ ]:
DATA_SLUG = "business-entity"   # <-- folder name under /kaggle/input
REPO = "https://github.com/stack-ajit/business_entity_resolution.git"
BRANCH = "main"
RUN_BLOCKING_EVAL = False       # full-scale recall report (~8 min); not needed for a submission

In [ ]:
!pip install -q anyascii==0.3.3 sparse-dot-topn==1.2.0 rapidfuzz lightgbm
import os, glob
!rm -rf /kaggle/working/repo && git clone -q -b $BRANCH $REPO /kaggle/working/repo
cand = [os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/{DATA_SLUG}/**/train", recursive=True)]     or [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/train", recursive=True)]
assert cand, "train/ folder not found - check DATA_SLUG"
os.environ["ER_DATA_DIR"] = cand[0]
os.environ["ER_WORK_DIR"] = "/kaggle/working"
print("data:", os.environ["ER_DATA_DIR"])
!ls $ER_DATA_DIR/train $ER_DATA_DIR/test
!nproc && free -g
%cd /kaggle/working/repo
!git log --oneline -1

## 1. (optional) Blocking recall on full train

In [ ]:
if RUN_BLOCKING_EVAL:
    !test -f /kaggle/working/sample/sample_ground_truth.tsv || python -u src/data/create_sample.py
    !python -u src/blocking/evaluate_blocking.py --top-k 100
else:
    # the training-pair builder needs the train index; build it without the evaluation
    !python -u src/blocking/build_train_index.py

## 2. Labelled training pairs (150K train S1 × top-50)

In [ ]:
!test -f /kaggle/working/cache/train_pairs.parquet || python -u src/matching/build_training_set.py --n-s1 150000 --top-k 50

## 3. Train matcher + tune selection rule for macro F0.5

In [ ]:
!test -f /kaggle/working/cache/model/lgb_matcher.txt || python -u src/matching/train_matcher.py
!cat /kaggle/working/cache/model/selection.json

## 4. Test inference → `output/matching_results.tsv` + `output/candidate_pairs.tsv`
Builds the test index, then streams 1.7M S1 in checkpointed batches and runs the official validator.

In [ ]:
!python -u src/matching/predict_test.py --top-k 50 --batch 100000
!ls -la /kaggle/working/output && head -3 /kaggle/working/output/matching_results.tsv

## 5. Free disk: keep only what later sessions need

In [ ]:
!rm -rf /kaggle/working/cache/train_index /kaggle/working/cache/test_index /kaggle/working/repo
!du -sh /kaggle/working/*